# Project Aura 2.0 — Notebook 06: Autonomous AI Agent

## Overview

Implements the autonomous agent pipeline: OBSERVATION → CONTEXT → DECISION → PERMISSION → EXECUTION → VERIFICATION → LOGGING. Deterministic, explainable, safe. Not an unrestricted LLM agent.

## Sections
1. Architecture
2. Imports & Config
3. Observation Schema
4. Context Manager
5. Decision Engine
6. Action Registry
7. Permission Manager
8. Action Executor
9. Verification
10. Decision Trace
11. Integration NB05
12. Integration NB04
13. Unit Tests
14. E2E Simulation
15. Export & Cleanup
16. Summary

In [ ]:
!pip install --quiet numpy
print("Dependencies installed for NB06")

## 1. Agent Architecture

Pipeline: OBSERVATION → CONTEXT → DECISION → PERMISSION → EXECUTION → VERIFICATION → LOGGING. Each stage is a separate reusable component.

## 2. Imports & Configuration

In [ ]:
import os, sys, json
for p in [os.getcwd(), "/content/project-aura"]:
    if p not in sys.path: sys.path.insert(0, p)

from src.core.config import load_config
from src.agents.observation import Observation, ObservationType, validate_observation
from src.agents.context import ContextManager, ContextSnapshot
from src.agents.decision_engine import DecisionEngine, Decision
from src.agents.action_registry import ActionRegistry
from src.agents.permission_manager import PermissionManager, PermissionRequest
from src.agents.executor import ActionExecutor, ExecutionResult
from src.agents.verification import VerificationSystem

config = load_config()
print(f"Config loaded: agent={config.agent.agent_name}")

## 3. Observation Schema

In [ ]:
obs = Observation(
    source="gesture_recognition",
    type=ObservationType.GESTURE,
    data={"gesture": "OPEN_PALM", "hand": "Right"},
    confidence=0.9,
)
print(f"Observation: {obs.observation_id[:8]}... type={obs.type.value}")
print(f"Data: {obs.data}")
print(f"Valid: {validate_observation(obs) == []}")
print(json.dumps(obs.to_dict(), indent=2))

In [ ]:
# Test malformed observation
bad = Observation(source="", type=ObservationType.GESTURE, data={})
bad.observation_id = ""
errors = validate_observation(bad)
print(f"Validation errors for bad obs: {len(errors)}")
for e in errors: print(f"  - {e}")

## 4. Context Manager

In [ ]:
ctx = ContextManager(ttl_seconds=30.0, max_events=50)

# Feed observations
ctx.update(Observation(source="inference", type=ObservationType.INFERENCE, data={"label":"cat"}, confidence=0.85))
ctx.update(Observation(source="gesture", type=ObservationType.GESTURE, data={"gesture":"OPEN_PALM"}, confidence=0.9))
ctx.update(Observation(source="gesture", type=ObservationType.GESTURE, data={"gesture":"FIST"}, confidence=0.88))

snap = ctx.snapshot()
print(f"Detected objects: {snap.detected_objects}")
print(f"Latest gesture: {snap.latest_gesture}, Previous: {snap.previous_gesture}")
print(f"Recent events: {len(snap.recent_events)}")
print(f"Expired: {snap.is_expired()}")

## 5. Decision Engine

In [ ]:
engine = DecisionEngine(confidence_threshold=0.7)

test_cases = [
    ("OPEN_PALM", 0.9, "pause"),
    ("THUMBS_UP", 0.9, "resume"),
    ("FIST", 0.9, "request_verification"),
    ("POINTING", 0.9, "update_visualization"),
    ("UNKNOWN", 0.9, "noop"),
    ("OPEN_PALM", 0.3, "noop"),
]

for gesture, conf, expected in test_cases:
    obs = Observation(source="test", type=ObservationType.GESTURE, data={"gesture": gesture}, confidence=conf)
    d = engine.decide(obs, ctx.snapshot())
    status = "OK" if d.selected_action == expected else "FAIL"
    print(f"  [{status}] {gesture} ({conf:.1f}) -> {d.selected_action}")

## 6. Action Registry

In [ ]:
registry = ActionRegistry()
print("Registered actions:")
for entry in registry.list_actions():
    status = "executable" if entry.is_executable() else "registered"
    sensitive = "sensitive" if entry.sensitive else "normal"
    print(f"  - {entry.name}: {status}, {sensitive}")

# Register a handler for testing
registry.register("pause", handler=lambda d: {"paused": True}, description="Pause sim")
registry.register("resume", handler=lambda d: {"resumed": True}, description="Resume sim")
print(f"\nPause executable: {registry.is_executable('pause')}")

## 7. Permission Manager

In [ ]:
pm = PermissionManager(registry)

# Non-sensitive action
req = PermissionRequest(action="pause", requesting_component="test")
res = pm.check_permission(req)
print(f"pause: granted={res.granted}, reason={res.reason}")

# Sensitive action (no confirmation)
req2 = PermissionRequest(action="save_screenshot", requesting_component="test")
res2 = pm.check_permission(req2)
print(f"save_screenshot: granted={res2.granted}, confirmation_required={res2.requires_confirmation}")

# Sensitive with confirmation
req3 = PermissionRequest(action="save_screenshot", requesting_component="test", confirmed_by_user=True)
res3 = pm.check_permission(req3)
print(f"save_screenshot (confirmed): granted={res3.granted}")

# Unknown action
req4 = PermissionRequest(action="dangerous_action", requesting_component="test")
res4 = pm.check_permission(req4)
print(f"unknown: granted={res4.granted}, reason={res4.reason}")

print(f"\nAudit summary: {pm.get_audit_summary()}")

## 8. Action Executor

In [ ]:
executor = ActionExecutor(registry, dry_run=True)

# Dry-run execution
req = PermissionRequest(action="pause", requesting_component="test")
perm = pm.check_permission(req)
result = executor.execute("pause", perm, data={"key": "val"})
print(f"Dry-run: succeeded={result.succeeded}, dry_run={result.dry_run}")

# Live execution
executor.set_dry_run(False)
result2 = executor.execute("pause", perm)
print(f"Live: succeeded={result2.succeeded}, data={result2.data}")

print(f"\nExecution summary: {executor.get_execution_summary()}")

## 9. Verification System

In [ ]:
verifier = VerificationSystem()

# Verify successful execution
record = verifier.verify(result2, expected_state={"paused": True})
print(f"Verified: {record.verified}, match: {record.match}")
print(f"Message: {record.message}")

print(f"\nVerification summary: {verifier.get_verification_summary()}")

## 10. Decision Trace

In [ ]:
print("=== COMPLETE DECISION TRACE ===")

# 1. Observation
obs = Observation(
    source="gesture_recognition",
    type=ObservationType.GESTURE,
    data={"gesture": "OPEN_PALM", "hand": "Right"},
    confidence=0.95,
)
print(f"\n1. OBSERVED: {obs.data} (confidence={obs.confidence})")

# 2. Context
ctx.update(obs)
snap = ctx.snapshot()
print(f"\n2. CONTEXT: latest={snap.latest_gesture}, prev={snap.previous_gesture}")

# 3. Decision
d = engine.decide(obs, snap)
print(f"\n3. DECIDED: action={d.selected_action}")
print(f"   Why: {d.explanation}")

# 4. Permission
req = PermissionRequest(action=d.selected_action, requesting_component="decision_engine")
perm = pm.check_permission(req)
print(f"\n4. PERMISSION: granted={perm.granted}")

# 5. Execution
if perm.granted:
    exec_result = executor.execute(d.selected_action, perm)
    print(f"\n5. EXECUTED: succeeded={exec_result.succeeded}")
    # 6. Verification
    v = verifier.verify(exec_result)
    print(f"\n6. VERIFIED: {v.verified}")
else:
    print(f"\n5. EXECUTION BLOCKED")

## 11. Integration with Notebook 05

In [ ]:
# Simulate consuming Notebook 05 gesture events
from src.vision.gesture_events import GestureEventGenerator

gen = GestureEventGenerator(source="nb05")
events = []
for g in ["OPEN_PALM", "OPEN_PALM", "FIST", "THUMBS_UP"]:
    events.append(gen.create_event(gesture=g, confidence=0.9, hand="Right"))

print(f"Consuming {len(events)} gesture events from NB05:")
for ev in events:
    obs = Observation(
        source="gesture_recognition",
        type=ObservationType.GESTURE,
        data={"gesture": ev.gesture, "hand": ev.hand},
        confidence=ev.confidence,
    )
    ctx.update(obs)
    d = engine.decide(obs, ctx.snapshot())
    print(f"  {ev.gesture} -> {d.selected_action} ({d.confidence:.2f})")

## 12. Integration with Notebook 04

In [ ]:
# Simulate Notebook 04 inference output
inference_obs = Observation(
    source="inference_engine",
    type=ObservationType.INFERENCE,
    data={"label": "airplane", "predicted_class": 0},
    confidence=0.87,
)
ctx.update(inference_obs)
snap = ctx.snapshot()
print(f"Detected: {snap.detected_objects}")

d = engine.decide(inference_obs, snap)
print(f"Decision for inference: {d.selected_action}")
print(f"Explanation: {d.explanation}")

## 13. Unit Tests

In [ ]:
print("Running agent tests...")

# Observation validation
o = Observation(source="test", type=ObservationType.GESTURE, data={"gesture":"FIST"}, confidence=0.9)
assert validate_observation(o) == []
print("  [OK] Observation validation")

# Decision rules
e = DecisionEngine()
c2 = ContextManager()
assert e.decide(Observation(source="t",type=ObservationType.GESTURE,data={"gesture":"OPEN_PALM"},confidence=0.9), c2.snapshot()).selected_action == "pause"
assert e.decide(Observation(source="t",type=ObservationType.GESTURE,data={"gesture":"UNKNOWN"},confidence=0.9), c2.snapshot()).selected_action == "noop"
print("  [OK] Decision rules")

# Permission rejection
r = ActionRegistry()
pm2 = PermissionManager(r)
assert pm2.check_permission(PermissionRequest(action="unknown_xyz", requesting_component="t")).granted is False
print("  [OK] Permission rejection")

# Dry-run execution
r.register("pause", handler=lambda d: {"ok": True})
ex = ActionExecutor(r, dry_run=True)
perm = pm2.check_permission(PermissionRequest(action="pause", requesting_component="t"))
assert ex.execute("pause", perm).dry_run is True
print("  [OK] Dry-run execution")

# Audit logging
assert len(pm2.get_audit_log()) > 0
print("  [OK] Audit logging")

print("\nAll tests passed")

## 14. End-to-End Simulation

In [ ]:
print("End-to-end agent simulation:")

# Fresh components
r = ActionRegistry()
r.register("pause", handler=lambda d: {"paused": True})
r.register("resume", handler=lambda d: {"resumed": True})
r.register("display_alert", handler=lambda d: {"shown": True})
r.register("record_event", handler=lambda d: {"recorded": True})
r.register("update_visualization", handler=lambda d: {"updated": True})
r.register("noop", handler=lambda d: {"nothing": True})
r.register("request_verification", handler=lambda d: {"requested": True})

c3 = ContextManager()
e3 = DecisionEngine()
pm3 = PermissionManager(r)
ex3 = ActionExecutor(r, dry_run=True)
v3 = VerificationSystem()

scenario = [
    ("OPEN_PALM", 0.95),
    ("FIST", 0.9),
    ("THUMBS_UP", 0.92),
    ("POINTING", 0.88),
    ("UNKNOWN", 0.0),
]

for gesture, conf in scenario:
    obs = Observation(source="sim", type=ObservationType.GESTURE, data={"gesture":gesture}, confidence=conf)
    c3.update(obs)
    d = e3.decide(obs, c3.snapshot())
    req = PermissionRequest(action=d.selected_action, requesting_component="engine")
    perm = pm3.check_permission(req)
    if perm.granted:
        res = ex3.execute(d.selected_action, perm)
        ver = v3.verify(res)
        print(f"  {gesture} -> {d.selected_action} (exec={res.succeeded}, verified={ver.verified})")
    else:
        print(f"  {gesture} -> {d.selected_action} (BLOCKED)")

print(f"\nContext gestures: {c3.latest_gesture()}")

## 15. Export & Cleanup

In [ ]:
import json, os
os.makedirs("output", exist_ok=True)

# Export audit log
with open("output/agent_audit.json", "w") as f:
    json.dump(pm.get_audit_log(), f, indent=2, default=str)
print("Audit log exported")

# Export decision trace
with open("output/agent_trace.json", "w") as f:
    json.dump({"decision_count": len(engine._rules)}, f, indent=2)
print("Trace exported")

print("Cleanup complete")

## 16. Final Summary

Implemented: observation schema, context manager, deterministic decision engine, action registry, permission manager, executor, verification, decision trace, NB05/NB04 integration. All tests pass.

**Next:** Notebook 07 (Deployment Preparation) or Notebook 08 (Monitoring).